In [129]:
import numpy as np
import pandas as pd
import math

# **Data Import**

In [130]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [131]:
data = pd.read_csv('/content/drive/MyDrive/CG-6_0251_G5-CE334.csv', skiprows=20)
data.columns = data.columns.str.replace('/', '')
data

,Station,Date,Time,CorrGrav,Line,StdDev,StdErr,RawGrav,X,Y,...,DriftCorr,MeasurDur,InstrHeight,LatUser,LonUser,ElevUser,LatGPS,LonGPS,ElevGPS,Corrections[drift-temp-na-tide-tilt]
0,FF1,14-06-06,09:58:05,2938.1049,0,0.0122,0.0022,2919.8542,-5.3,4.4,...,18.3922,30,0,26.515766,80.230446,101.0,26.515760,80.230431,101.4,11011
1,FF1,14-06-06,09:58:35,2938.1060,0,0.0110,0.0020,2919.8561,-3.1,4.9,...,18.3922,30,0,26.515766,80.230446,101.0,26.515820,80.230408,101.7,11011
2,FF1,14-06-06,09:59:05,2938.1043,0,0.0111,0.0020,2919.8545,-4.5,5.1,...,18.3922,30,0,26.515766,80.230446,101.0,26.515810,80.230431,101.0,11011
3,TF1,14-06-06,10:04:28,2936.0332,0,0.0114,0.0021,2917.7870,-5.2,4.6,...,18.3922,30,0,26.515877,80.230370,112.5,26.515865,80.230385,111.4,11011
4,TF1,14-06-06,10:04:58,2936.0323,0,0.0124,0.0023,2917.7858,-7.2,5.3,...,18.3922,30,0,26.515877,80.230370,112.5,26.515877,80.230408,112.5,11011
5,TF1,14-06-06,10:05:28,2936.0331,0,0.0151,0.0028,2917.7872,-7.2,4.6,...,18.3922,30,0,26.515877,80.230370,112.5,26.515862,80.230408,112.3,11011
6,FF2,14-06-06,10:10:01,2938.0923,0,0.0112,0.0021,2919.8495,1.4,3.1,...,18.3922,30,0,26.515919,80.230408,113.3,26.515865,80.230408,110.1,11011
7,FF2,14-06-06,10:10:31,2938.0895,0,0.0119,0.0022,2919.8473,2.0,2.2,...,18.3922,30,0,26.515919,80.230408,113.3,26.515820,80.230423,109.3,11011
8,FF2,14-06-06,10:11:01,2938.0904,0,0.0138,0.0025,2919.8486,2.6,3.1,...,18.3922,30,0,26.515919,80.230408,113.3,26.515800,80.230423,109.4,11011
9,TF2,14-06-06,10:16:21,2936.0212,0,0.0172,0.0031,2917.7815,5.6,8.0,...,18.3922,30,0,26.515837,80.230446,115.1,26.515818,80.230431,114.0,11011


# **Average values of gravity, standered deviation and time**

In [132]:
grouped = data.groupby("Station")

def hms_to_decimal(hms):
    h, m, s = map(int, hms.split(':'))
    return h + m/60 + s/3600
data["Time_decimal"] = data["Time"].apply(hms_to_decimal)

# Mean gravity and time
mean_gravity = grouped["CorrGrav"].mean()
mean_time = grouped["Time_decimal"].mean()

# RMS of standard deviation
rms_std = np.sqrt(grouped["StdDev"].apply(lambda x: np.mean(x**2)))

# Create averaged dataframe
avg_data = pd.DataFrame({
    "g": mean_gravity,
    "t": mean_time,
    "std": rms_std
})

avg_data = avg_data.reset_index()

avg_data

,Station,g,t,std
0,5F1,2933.822767,10.360278,0.018492
1,5F2,2933.809733,10.520278,0.017820
2,5F3,2933.791933,10.691944,0.018626
3,7F1,2931.627800,10.611389,0.018309
4,7F2,2931.614500,10.773889,0.021903
5,FF1,2938.105067,9.976389,0.011446
6,FF2,2938.090733,10.175278,0.012349
7,TF1,2936.032867,10.082778,0.013061
8,TF2,2936.020333,10.280833,0.019126
9,TF3,2936.004767,10.439444,0.019580


In [133]:
# now we will sort the matrix in incresing order of time
avg_data = avg_data.sort_values(by="t")
avg_data.reset_index(drop=True, inplace=True)
avg_data

,Station,g,t,std
0,FF1,2938.105067,9.976389,0.011446
1,TF1,2936.032867,10.082778,0.013061
2,FF2,2938.090733,10.175278,0.012349
3,TF2,2936.020333,10.280833,0.019126
4,5F1,2933.822767,10.360278,0.018492
5,TF3,2936.004767,10.439444,0.019580
6,5F2,2933.809733,10.520278,0.017820
7,7F1,2931.627800,10.611389,0.018309
8,5F3,2933.791933,10.691944,0.018626
9,7F2,2931.614500,10.773889,0.021903


# **General Equations**

In [134]:
# The general equation will be given by y_n(t) = g_n + b + d.t_k + c

# **General Difference Equation**

In [135]:
# General difference equation will be given by--> del y_n(t) = del g_n + d.(t_n - t_n-1) + e

In [136]:
# first station is FF1
ref = avg_data.iloc[0]

L = []  # L is the vector that consists of relative gravity of other point with respect to first point on first floor
delta_t = []  # it is the vector that tells relative time difference between points
var_L = []  # it is the variance vector that store the variance of all point with respect to first point

for i in range(1, len(avg_data)):

    dy = avg_data.loc[i, "g"] - ref["g"]
    dt = avg_data.loc[i, "t"] - ref["t"]

    # variance propagation (independent)
    var = avg_data.loc[i, "std"]**2 + ref["std"]**2

    L.append(dy)
    delta_t.append(dt)
    var_L.append(var)

L = np.array(L).reshape(-1,1)
delta_t = np.array(delta_t).reshape(-1,1)
var_L = np.array(var_L)
diff_mat = np.hstack((L, delta_t, var_L.reshape(-1,1)))
print('   L               ', 'delta_t          ', 'var_L         ')
print(diff_mat)


   L                delta_t           var_L         
[[-2.07220000e+00  1.06388889e-01  3.01593333e-04]
 [-1.43333333e-02  1.98888889e-01  2.83513333e-04]
 [-2.08473333e+00  3.04444444e-01  4.96836667e-04]
 [-4.28230000e+00  3.83888889e-01  4.72970000e-04]
 [-2.10030000e+00  4.63055556e-01  5.14383333e-04]
 [-4.29533333e+00  5.43888889e-01  4.48566667e-04]
 [-6.47726667e+00  6.35000000e-01  4.66253333e-04]
 [-4.31313333e+00  7.15555556e-01  4.77953333e-04]
 [-6.49056667e+00  7.97500000e-01  6.10770000e-04]]


# **Desgin Matrix A**

In [137]:
# DESIGN MATRIX A (9x4)
# Unknowns: g3, g5, g7, d
# 4 unknown parameters (3 gravity floors + drift)

n_obs = len(L)
A = np.zeros((n_obs, 4))

for i in range(n_obs):
    if i == 1:
      A[i, 0] = 0

    if i in [0, 2, 4]:   # del_g_3
      A[i, 0] = 1

    if i in [3, 5, 7]:   # del_g_5
      A[i, 1] = 1

    if i in [6, 8]:    # del_g_7
      A[i,2] = 1  # gravity unknowns

    A[i, 3] = delta_t[i]  # drift column
print(f'the desgin matrix "A" is given by')
print(A)

the desgin matrix "A" is given by
[[1.         0.         0.         0.10638889]
 [0.         0.         0.         0.19888889]
 [1.         0.         0.         0.30444444]
 [0.         1.         0.         0.38388889]
 [1.         0.         0.         0.46305556]
 [0.         1.         0.         0.54388889]
 [0.         0.         1.         0.635     ]
 [0.         1.         0.         0.71555556]
 [0.         0.         1.         0.7975    ]]


/tmp/ipython-input-2594726616.py:21: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  A[i, 3] = delta_t[i]  # drift column


# **Variance Propagation**

In [138]:
var_matrix = np.zeros((n_obs, n_obs))
for i in range(n_obs):
    var_matrix[i, i] = var_L[i]
var_matrix

array([[0.00030159, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.00028351, 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.00049684, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.00047297, 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.00051438,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.00044857, 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.00046625, 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.00047795, 0.        ],


# **Weight Matrix**

In [139]:
P = np.diag(1/var_L)
print('the weight matrix P is given by ')
print(P)

the weight matrix P is given by 
[[3315.72315922    0.            0.            0.            0.
     0.            0.            0.            0.        ]
 [   0.         3527.17097373    0.            0.            0.
     0.            0.            0.            0.        ]
 [   0.            0.         2012.73389645    0.            0.
     0.            0.            0.            0.        ]
 [   0.            0.            0.         2114.29900417    0.
     0.            0.            0.            0.        ]
 [   0.            0.            0.            0.         1944.07543013
     0.            0.            0.            0.        ]
 [   0.            0.            0.            0.            0.
  2229.32302891    0.            0.            0.        ]
 [   0.            0.            0.            0.            0.
     0.         2144.75678458    0.            0.        ]
 [   0.            0.            0.            0.            0.
     0.            0.         2092

# **Normal Equation**

In [140]:
# the noraml equation is given by N.x = U
N = A.T @ P @ A
U = A.T @ P @ L
print('the normal vector N is given by')
print(N)
print('\n')
print('the vector U is given by')
print(U)
print('\n')


# the unknown vector X will be
X = (np.linalg.inv(N)) @ U
print('The unknowm vector X is as follow')
print(X)

the normal vector N is given by
[[7272.5324858     0.            0.         1865.73668403]
 [   0.         6435.87650701    0.         3521.28423312]
 [   0.            0.         3782.0343195  2667.6493923 ]
 [1865.73668403 3521.28423312 2667.6493923  4728.92257999]]


the vector U is given by
[[-15149.9966015 ]
 [-27653.92065572]
 [-24519.02062107]
 [-36346.8130965 ]]


The unknowm vector X is as follow
[[-2.06268059]
 [-4.25311861]
 [-6.42666229]
 [-0.07990679]]


# **Observation Equation**

In [141]:
# AX - L
#Adjusted Observation
Adj_obs = A @ X
print('The adjusted observation is given by')
print(Adj_obs)
print('\n')

# Residual vector
Res_vec = Adj_obs - L
print('The residual vector is given by')
print(Res_vec)

The adjusted observation is given by
[[-2.07118178]
 [-0.01589257]
 [-2.08700776]
 [-4.28379394]
 [-2.09968187]
 [-4.29657902]
 [-6.4774031 ]
 [-4.31029636]
 [-6.49038795]]


The residual vector is given by
[[ 0.00101822]
 [-0.00155924]
 [-0.00227443]
 [-0.00149394]
 [ 0.00061813]
 [-0.00124569]
 [-0.00013643]
 [ 0.00283698]
 [ 0.00017872]]


# **Priori and Posteriori Variance**

In [142]:
# Number of observations = 9
# Number of unknowns = 4
# Redundancy = 5

r = 5
V = Res_vec
VTPV = V.T @ P @ V
sigma_priori = 1.0  # Assuming unit variance
sigma_posteriori = VTPV / r
variance_ratio = sigma_posteriori / sigma_priori
print("Variance Ratio:", variance_ratio)
print("Priori Variance:", sigma_priori)
print("Posteriori Variance:", sigma_posteriori)

Variance Ratio: [[0.0096555]]
Priori Variance: 1.0
Posteriori Variance: [[0.0096555]]


# **Checks**

In [143]:
tol = 1e-6

# Orthogonality Condition ATPV = 0

ATPV = A.T @ P @ V
ortho_check = np.max(np.abs(ATPV))

print(f"   Orthogonality (max|ATPV|): {ortho_check} ")
print(f"   ATPV = {ATPV.T}")
print(f"   Pass: {ortho_check < tol}")



   Orthogonality (max|ATPV|): 5.331524345896922e-11 
   ATPV = [[-2.30508945e-11 -3.02828873e-11 -4.20621131e-11 -5.33152435e-11]]
   Pass: True


In [144]:
#  VᵀPV consistency using 3 standered equations

VTPV_m1 = V.T @ P @ V                     # Direct
VTPV_m2 = -V.T @ P @ L                    # Residual form
VTPV_m3 = -(U.T @ X) + (L.T @ P @ L)  # Normal equation form

print("   VTPV Consistency:")
print(f"   Method 1 (Direct):            {VTPV_m1}")
print(f"   Method 2 (Residual):          {VTPV_m2}")
print(f"   Method 3 (Normal equation):   {VTPV_m3}")

diff_12 = abs(VTPV_m1 - VTPV_m2)
diff_13 = abs(VTPV_m1 - VTPV_m3)

print("\n   Differences:")
print(f"   |Method1 − Method2| = {diff_12}")
print(f"   |Method1 − Method3| = {diff_13}")

print(f"\n   Pass: {(diff_12 < tol) and (diff_13 < tol)}")

   VTPV Consistency:
   Method 1 (Direct):            [[0.04827748]]
   Method 2 (Residual):          [[0.04827748]]
   Method 3 (Normal equation):   [[0.04827748]]

   Differences:
   |Method1 − Method2| = [[4.50919239e-10]]
   |Method1 − Method3| = [[4.45357022e-10]]

   Pass: [[ True]]


# **Final Gravity Values**

In [145]:
# The final results of gravity of 3rd 5th and 7th floor with respect to first floor are

print('       The final results are             \n')

g_3_final = X[0] + ref['g']
g_5_final = X[1] + ref['g']
g_7_final = X[2] + ref['g']
d_final = X[3]

final_results = pd.DataFrame({
    'Floor': ['Third', 'Fifth', 'Seventh'],
    'Adjusted_Gravity (mGal)': [g_3_final, g_5_final, g_7_final]})

print(final_results.to_string(index=False))

print('\n')

print(f"Estimated Linear Drift: {d_final} mGal/hr")

       The final results are             

  Floor Adjusted_Gravity (mGal)
  Third     [2936.042386079119]
  Fifth      [2933.85194805531]
Seventh    [2931.6784043784887]


Estimated Linear Drift: [-0.07990679] mGal/hr
